# GAT (Graph Attention Network) on Amazon Computers Dataset

Node classification using GAT variants with full-batch training.

**Dataset:** Amazon Computers (13,752 nodes, 491,722 edges, 767 features, 10 classes)

## GAT v1 run

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.datasets import Amazon
import torch_geometric.transforms as T
from torch_geometric.nn import GATConv
from torch.nn import BatchNorm1d

# 1. Load Amazon Computers Dataset and Masks
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Amazon(root='./data/Amazon', name='Computers', transform=transform)
data = dataset[0]

# 2. Define the GAT Architecture
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super().__init__()
        # Layer 1: Multi-head attention.
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        self.bn1 = BatchNorm1d(hidden_channels * heads)
        
        # Layer 2: Output layer. concat=False averages the heads.
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=heads, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# 3. Initialize Device, Model, and Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GAT(dataset.num_node_features, hidden_channels=64, out_channels=dataset.num_classes, heads=8).to(device)
data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)

# 4. Training and Testing Functions (Full-Batch)
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    pred_cpu = pred.cpu().numpy()
    y_cpu = data.y.cpu().numpy()
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        m = mask.cpu().numpy()
        accs.append(f1_score(y_cpu[m], pred_cpu[m], average='macro'))
    return accs

# 5. Execution Loop
print("Starting 2-Layer GAT v1 training with Scheduler...")
for epoch in range(1, 201):
    loss = train()
    train_f1, val_f1, test_f1 = test()
    scheduler.step(val_f1)
    
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train F1: {train_f1:.4f}, Val F1: {val_f1:.4f}')

print(f'\nTraining Complete!')
print(f'Final Test F1uracy: {test_f1:.4f}')

## GAT v2 Baseline

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.datasets import Amazon
import torch_geometric.transforms as T
from torch_geometric.nn import GATConv, GATv2Conv
from torch.nn import BatchNorm1d

# 1. Load Amazon Computers Dataset and Masks
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Amazon(root='./data/Amazon', name='Computers', transform=transform)
data = dataset[0]

# 2. Define the GATv2 Architecture
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads)
        self.bn1 = BatchNorm1d(hidden_channels * heads)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=heads, concat=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# 3. Initialize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GAT(dataset.num_node_features, hidden_channels=64, out_channels=dataset.num_classes, heads=8).to(device)
data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)

# 4. Training and Testing Functions (Full-Batch)
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    pred_cpu = pred.cpu().numpy()
    y_cpu = data.y.cpu().numpy()
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        m = mask.cpu().numpy()
        accs.append(f1_score(y_cpu[m], pred_cpu[m], average='macro'))
    return accs

# 5. Execution Loop
print("Starting 2-Layer GAT v2 training with Scheduler...")
for epoch in range(1, 201):
    loss = train()
    train_f1, val_f1, test_f1 = test()
    scheduler.step(val_f1)
    
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train F1: {train_f1:.4f}, Val F1: {val_f1:.4f}')

print(f'\nTraining Complete!')
print(f'Final Test F1uracy: {test_f1:.4f}')

## GAT v2 with MLP head

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.datasets import Amazon
import torch_geometric.transforms as T
from torch_geometric.nn import GATv2Conv, BatchNorm

# 1. Load Amazon Computers Dataset
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Amazon(root='./data/Amazon', name='Computers', transform=transform)
data = dataset[0]

# 2. GATv2 Architecture with MLP Head
class GATv2WithMLP(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super().__init__()
        
        # --- GNN Encoder (Dynamic Attention) ---
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads)
        self.bn1 = BatchNorm(hidden_channels * heads)
        
        self.conv2 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, concat=False)
        self.bn2 = BatchNorm(hidden_channels)

        # --- MLP Task Head ---
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(hidden_channels, out_channels)
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)
        x = self.mlp(x)
        return x

# 3. Initialize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GATv2WithMLP(
    in_channels=dataset.num_node_features,
    hidden_channels=32,
    out_channels=dataset.num_classes,
    heads=4
).to(device)

data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

# 4. Training and Testing (Full-Batch)
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    pred_cpu = pred.cpu().numpy()
    y_cpu = data.y.cpu().numpy()
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        m = mask.cpu().numpy()
        accs.append(f1_score(y_cpu[m], pred_cpu[m], average='macro'))
    return accs

# 5. Execution Loop
print(f"Benchmarking GATv2 + MLP Head on {device}...")
best_val_f1 = 0.0

for epoch in range(1, 201):
    loss = train()
    train_f1, val_f1, test_f1 = test()
    scheduler.step(val_f1)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'Amazon_GATv2_bestmodel.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Val F1: {val_f1:.4f}, Test F1: {test_f1:.4f}')

# Final evaluation using best saved weights
model.load_state_dict(torch.load('Amazon_GATv2_bestmodel.pt', weights_only=True))
final_train_f1, final_val_f1, final_test_f1 = test()

print(f'\nTraining Complete!')
print(f'Best Validation Macro-F1: {best_val_f1:.4f}')
print(f'Final Test F1uracy (from best model): {final_test_f1:.4f}')

### ADDITION: GAT v1 with Residual Dense Head

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.datasets import Amazon
import torch_geometric.transforms as T
from torch_geometric.nn import GATConv
from torch.nn import BatchNorm1d

# ──────────────────────────────────────────────────────────────────────
# 1. Load Amazon Computers Dataset
# ──────────────────────────────────────────────────────────────────────
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Amazon(root='./data/Amazon', name='Computers', transform=transform)
data = dataset[0]

# ──────────────────────────────────────────────────────────────────────
# 2. Residual Dense Head (DenseNet-style classifier)
# ──────────────────────────────────────────────────────────────────────
class ResidualDenseHead(nn.Module):
    """
    A 3-block dense head with residual skip connections.
    Given encoder output h (dim = in_dim):
        Block 1:  d1 = ELU(BN(Linear(h)))                  -> dense_dim
        Block 2:  d2 = ELU(BN(Linear(h || d1)))             -> dense_dim
        Block 3:  d3 = ELU(BN(Linear(h || d1 || d2)))       -> dense_dim
        Output :  logits = Linear(h || d1 || d2 || d3)       -> out_dim
    '||' denotes concatenation.
    """
    def __init__(self, in_dim, dense_dim, out_dim, drop_p=0.5):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, dense_dim)
        self.bn1 = nn.BatchNorm1d(dense_dim)
        self.fc2 = nn.Linear(in_dim + dense_dim, dense_dim)
        self.bn2 = nn.BatchNorm1d(dense_dim)
        self.fc3 = nn.Linear(in_dim + dense_dim * 2, dense_dim)
        self.bn3 = nn.BatchNorm1d(dense_dim)
        self.out_proj = nn.Linear(in_dim + dense_dim * 3, out_dim)
        self.drop_p = drop_p

    def forward(self, h):
        d1 = F.elu(self.bn1(self.fc1(h)))
        d1 = F.dropout(d1, p=self.drop_p, training=self.training)
        cat1 = torch.cat([h, d1], dim=-1)
        d2 = F.elu(self.bn2(self.fc2(cat1)))
        d2 = F.dropout(d2, p=self.drop_p, training=self.training)
        cat2 = torch.cat([h, d1, d2], dim=-1)
        d3 = F.elu(self.bn3(self.fc3(cat2)))
        d3 = F.dropout(d3, p=self.drop_p, training=self.training)
        cat3 = torch.cat([h, d1, d2, d3], dim=-1)
        return self.out_proj(cat3)

# ──────────────────────────────────────────────────────────────────────
# 3. GAT Encoder + Residual Dense Head
# ──────────────────────────────────────────────────────────────────────
class GATWithDenseHead(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 heads=8, dense_dim=128, head_drop=0.5):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        self.bn1 = BatchNorm1d(hidden_channels * heads)
        self.res1 = nn.Linear(in_channels, hidden_channels * heads)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=heads, concat=False)
        self.bn2 = BatchNorm1d(hidden_channels)
        self.res2 = nn.Linear(hidden_channels * heads, hidden_channels)
        self.head = ResidualDenseHead(in_dim=hidden_channels, dense_dim=dense_dim, out_dim=out_channels, drop_p=head_drop)

    def forward(self, x, edge_index):
        identity = self.res1(x)
        x = self.conv1(x, edge_index) + identity
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        identity = self.res2(x)
        x = self.conv2(x, edge_index) + identity
        x = self.bn2(x)
        x = F.elu(x)
        return self.head(x)

# ──────────────────────────────────────────────────────────────────────
# 4. Initialization
# ──────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GATWithDenseHead(
    in_channels=dataset.num_node_features,   # 767
    hidden_channels=64,
    out_channels=dataset.num_classes,        # 10
    heads=8, dense_dim=128, head_drop=0.5,
).to(device)
data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)

# ──────────────────────────────────────────────────────────────────────
# 5. Training (Full-Batch)
# ──────────────────────────────────────────────────────────────────────
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    pred_cpu = pred.cpu().numpy()
    y_cpu = data.y.cpu().numpy()
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        m = mask.cpu().numpy()
        accs.append(f1_score(y_cpu[m], pred_cpu[m], average='macro'))
    return accs

# ──────────────────────────────────────────────────────────────────────
# 6. Execution Loop
# ──────────────────────────────────────────────────────────────────────
print(f"Starting GAT + Residual Dense Head Training on {device}...")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
best_val_f1 = 0.0
best_test_f1 = 0.0

for epoch in range(1, 201):
    loss = train()
    train_f1, val_f1, test_f1 = test()
    scheduler.step(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_test_f1 = test_f1
        torch.save(model.state_dict(), 'Amazon_GAT_denseHead_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train: {train_f1:.4f}, Val: {val_f1:.4f}, '
              f'Test: {test_f1:.4f}, Best Val: {best_val_f1:.4f}')

print(f"\nFinal Best Validation Macro-F1: {best_val_f1:.4f}")
print(f"Test F1uracy at Best Val:      {best_test_f1:.4f}")